In [ ]:
import os, glob, random
import numpy as np
import pandas as pd
import tensorflow as tf


In [ ]:

dataset_path = "/path/to/local/resource"

csv_path   = os.path.join(dataset_path, "Data_Entry_2017.csv")
trainval_txt = os.path.join(dataset_path, "train_val_list.txt")
test_txt     = os.path.join(dataset_path, "test_list.txt")

df = pd.read_csv(csv_path)
df["Image Index"] = df["Image Index"].astype(str).str.strip()
df["Finding Labels"] = df["Finding Labels"].astype(str)

# splitting data as nih provided
with open(trainval_txt, "r") as f:
    trainval_list = set([line.strip() for line in f if line.strip()])

with open(test_txt, "r") as f:
    test_list = set([line.strip() for line in f if line.strip()])

print("CSV rows:", len(df))
print("Train/Val list:", len(trainval_list))
print("Test list:", len(test_list))


In [ ]:
#indexing images

image_dirs = []
for d in os.listdir(dataset_path):
    if d.startswith("images_"):
        inner = os.path.join(dataset_path, d, "images")
        if os.path.isdir(inner):
            image_dirs.append(inner)

print("Image folders:", len(image_dirs))

image_path_map = {}
for img_dir in image_dirs:
    for p in glob.glob(os.path.join(img_dir, "*.png")):
        image_path_map[os.path.basename(p)] = p

df["Path"] = df["Image Index"].map(image_path_map)

missing = df["Path"].isna().sum()
print("Missing images:", missing)
assert missing == 0, "❌ Some images are missing from path mapping!"


In [ ]:
# Binary label:
# Positive if Mass or Nodule
# Negative if No Finding only
df["is_positive"] = df["Finding Labels"].apply(lambda s: int(("Mass" in s) or ("Nodule" in s)))
df["is_negative"] = df["Finding Labels"].apply(lambda s: int(s.strip() == "No Finding"))

# Limit to the cases we want (positive or negative only)
df_bin = df[(df["is_positive"] == 1) | (df["is_negative"] == 1)].copy()

#Final label: 1 = positive, 0 = negative
df_bin["label"] = df_bin["is_positive"].astype(int)

# Official Split App
df_trainval = df_bin[df_bin["Image Index"].isin(trainval_list)].copy()
df_test     = df_bin[df_bin["Image Index"].isin(test_list)].copy()

print("Binary eligible:", len(df_bin))
print("Train/Val bin:", len(df_trainval))
print("Test bin:", len(df_test))

# sanity check
assert df_trainval["Image Index"].isin(test_list).sum() == 0, "❌ Leakage: trainval contains test!"


In [ ]:
SEED = 42
np.random.seed(SEED)

# maximum limit
N_PER_CLASS = 20000

pos = df_trainval[df_trainval["label"] == 1]
neg = df_trainval[df_trainval["label"] == 0]

print("TrainVal positives:", len(pos))
print("TrainVal negatives:", len(neg))

# Take the greatest possible balance (without failure)
TARGET = min(N_PER_CLASS, len(pos), len(neg))
print("Using TARGET per class:", TARGET)

# If the TARGET is too small, we stop and change the positive definition
assert TARGET > 0, "❌ No samples found for one of the classes!"

pos_sample = pos.sample(n=TARGET, random_state=SEED)
neg_sample = neg.sample(n=TARGET, random_state=SEED)

df_bal = pd.concat([pos_sample, neg_sample]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Split train/val (90/10)
val_frac = 0.1
val_size = int(len(df_bal) * val_frac)

df_val = df_bal.iloc[:val_size].copy()
df_train = df_bal.iloc[val_size:].copy()

print("Balanced Train:", len(df_train))
print("Balanced Val:", len(df_val))
print("Train pos ratio:", df_train["label"].mean())
print("Val pos ratio:", df_val["label"].mean())


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

def parse_png(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    label = tf.cast(label, tf.float32)
    return img, label

def make_ds(df_in, training=False):
    paths = df_in["Path"].values
    labels = df_in["label"].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(df_in), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(parse_png, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(df_train, training=True)
val_ds   = make_ds(df_val, training=False)
test_ds  = make_ds(df_test, training=False)

print("✅ train/val/test datasets ready")
